# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Construir rankings TF-IDF para cuatro consultas.

### 1. Recuperar y revisar el corpus

Se aplicó la misma tokenización y preprocesamiento del notebook anterior


In [1]:
import re
import unicodedata
from collections import Counter

import nltk
import numpy as np
import pandas as pd
from nltk.corpus import gutenberg
from scipy.sparse import csr_matrix
from IPython.display import display


nltk.download('gutenberg', quiet=True)
identificadores = gutenberg.fileids()

patron_token = re.compile(r"\w+(?:'\w+)*", flags=re.UNICODE)

def normalizar_tokenizar(texto):
    texto = texto.lower()
    texto = texto.replace('\u2019', "'")
    texto = texto.replace('\u2018', "'")
    texto = texto.replace('_', ' ')
    texto = unicodedata.normalize('NFC', texto)
    return patron_token.findall(texto)

textos_originales = {}
tokens_por_documento = {}
for doc_id in identificadores:
    texto = gutenberg.raw(doc_id)
    textos_originales[doc_id] = texto
    tokens_por_documento[doc_id] = normalizar_tokenizar(texto)

def titulo_de(doc_id):
    inicio = textos_originales[doc_id][:2000]
    for linea in inicio.splitlines():
        linea = linea.strip()
        if linea.startswith('['):
            cierre = linea.find(']')
            if cierre > 1:
                titulo = linea[1:cierre]
                fin_titulo = titulo.lower().find(' by ')
                if fin_titulo >= 0:
                    titulo = titulo[:fin_titulo]
                return titulo.strip()
    raise ValueError('Sin titulo: ' + doc_id)

longitudes = {}
filas_corpus = []
vocabulario_corpus = set()
for doc_id in identificadores:
    tokens = tokens_por_documento[doc_id]
    longitudes[doc_id] = len(tokens)
    terminos_unicos = set(tokens)
    vocabulario_corpus.update(terminos_unicos)
    filas_corpus.append({
        'Identificador de documento': doc_id,
        'Titulo': titulo_de(doc_id),
        'Numero de terminos unicos': len(terminos_unicos),
    })

display(pd.DataFrame(filas_corpus))
valores_longitud = list(longitudes.values())
print('Documentos:', len(identificadores))
print('Tamano del vocabulario:', len(vocabulario_corpus))
print('Longitud minima:', min(valores_longitud))
print('Longitud media:', round(np.mean(valores_longitud), 2))
print('Longitud maxima:', max(valores_longitud))


,Identificador de documento,Titulo,Numero de terminos unicos
0,austen-emma.txt,Emma,7211
1,austen-persuasion.txt,Persuasion,5860
2,austen-sense.txt,Sense and Sensibility,6431
3,bible-kjv.txt,The King James Bible,12965
4,blake-poems.txt,Poems,1544
5,bryant-stories.txt,Stories to Tell to Children,3915
6,burgess-busterbrown.txt,The Adventures of Buster Bear,1549
7,carroll-alice.txt,Alice's Adventures in Wonderland,2630
8,chesterton-ball.txt,The Ball and The Cross,8340
9,chesterton-brown.txt,The Wisdom of Father Brown,7884


Documentos: 18
Tamano del vocabulario: 43824
Longitud minima: 6839
Longitud media: 121224.33
Longitud maxima: 852256


### 2. Implementación de Funciones

Se calcula tres medidas por término, TF cuenta sus apariciones en un libro, DF cuenta en cuántos libros aparece, sin importar cuántas veces se repita dentro de cada uno y el IDF asigna menor peso a las palabras presentes en muchos libros. Multiplicando TF por IDF se obtiene el peso TF-IDF. Mostramos en la tabla seis ejemplos concretos, incluido uno con TF igual a cero.


In [2]:
def frecuencias_tf(tokens_por_doc):
    frecuencias = {}
    for doc_id, tokens in tokens_por_doc.items():
        frecuencias[doc_id] = Counter(tokens)
    return frecuencias

def frecuencias_df(frecuencias):
    df = Counter()
    for conteos in frecuencias.values():
        df.update(conteos.keys())
    return df

def idf_sin_suavizar(df, n_documentos):
    idf = {}
    for termino, frecuencia in df.items():
        idf[termino] = float(np.log(n_documentos / frecuencia) + 1)
    return idf

tf_por_documento = frecuencias_tf(tokens_por_documento)
df_por_termino = frecuencias_df(tf_por_documento)
idf_por_termino = idf_sin_suavizar(df_por_termino, len(identificadores))

ejemplos = [
    ('the', 'austen-emma.txt'),
    ('whale', 'melville-moby_dick.txt'),
    ('whale', 'carroll-alice.txt'),
    ('alice', 'carroll-alice.txt'),
    ('god', 'bible-kjv.txt'),
    ('love', 'shakespeare-hamlet.txt'),
]
filas_pesos = []
for termino, doc_id in ejemplos:
    tf = tf_por_documento[doc_id][termino]
    df = df_por_termino[termino]
    idf = idf_por_termino[termino]
    filas_pesos.append({
        'Termino': termino, 'Documento': doc_id,
        'TF': tf, 'DF': df, 'IDF': idf, 'TF-IDF': tf * idf,
    })

display(pd.DataFrame(filas_pesos).round(6))
print('Cada peso TF-IDF es TF * IDF. Si TF es cero, el peso tambien es cero.')

,Termino,Documento,TF,DF,IDF,TF-IDF
0,the,austen-emma.txt,5204,18,1.000000,5204.000000
1,whale,melville-moby_dick.txt,1101,6,2.098612,2310.572130
2,whale,carroll-alice.txt,0,6,2.098612,0.000000
3,alice,carroll-alice.txt,386,3,2.791759,1077.619155
4,god,bible-kjv.txt,4446,16,1.117783,4969.663377
5,love,shakespeare-hamlet.txt,0,15,1.182322,0.000000


Cada peso TF-IDF es TF * IDF. Si TF es cero, el peso tambien es cero.


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

Se ordena alfabéticamente el vocabulario para que cada término tenga siempre la misma columna y se asigna una fila a cada libro. También se construye una matriz dispersa de frecuencias TF, que guarda solo las entradas con valor distinto de cero, y multiplica cada columna por su IDF. El resultado es la matriz TF-IDF


In [3]:
terminos = sorted(df_por_termino)
columna_de = {}
for j, termino in enumerate(terminos):
    columna_de[termino] = j

filas = []
columnas = []
conteos = []
for i, doc_id in enumerate(identificadores):
    for termino, tf in tf_por_documento[doc_id].items():
        filas.append(i)
        columnas.append(columna_de[termino])
        conteos.append(tf)

matriz_tf = csr_matrix(
    (np.array(conteos, dtype=float), (filas, columnas)),
    shape=(len(identificadores), len(terminos)),
)
vector_idf = np.zeros(len(terminos))
for j, termino in enumerate(terminos):
    vector_idf[j] = idf_por_termino[termino]

matriz_manual = matriz_tf.multiply(vector_idf).tocsr()
print('Dimensiones de la matriz TF-IDF:', matriz_manual.shape)
print('Pesos no nulos:', matriz_manual.nnz)

Dimensiones de la matriz TF-IDF: (18, 43824)
Pesos no nulos: 124322


La matriz tiene 788,832 celdas y solo 124,322 son distintas de cero (un 15.8 % aproximadamente), donde el 84.2 % son ceros porque cada libro usa una fracción pequeña del vocabulario total.

### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Con TfidfVectorizer se usan los tokens ya procesados, el mismo vocabulario y la misma fórmula de IDF que en el cálculo manual anterior. Así, cada posición de su matriz corresponde al mismo libro y término que en la matriz anterior. Se calcula la mayor diferencia absoluta entre pesos equivalentes y se detiene la ejecución si supera $10^{-9}$, esto ya que una diferencia menor se considera compatible con el redondeo numérico.


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

def conservar_tokens(tokens):
    return tokens

documentos_procesados = []
for doc_id in identificadores:
    documentos_procesados.append(tokens_por_documento[doc_id])

vectorizador = TfidfVectorizer(
    analyzer=conservar_tokens,
    vocabulary=columna_de,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
)
matriz_sklearn = vectorizador.fit_transform(documentos_procesados)

diferencia_maxima = float(abs(matriz_manual - matriz_sklearn).max())
tolerancia = 1e-9
print('Diferencia maxima absoluta entre matrices:', diferencia_maxima)
if diferencia_maxima > tolerancia:
    raise ValueError('No coinciden las matrices TF-IDF')
print('Matrices TF-IDF iguales')

Diferencia maxima absoluta entre matrices: 0.0
Matrices TF-IDF iguales


Se utiliza `analyzer=conservar_tokens` y `vocabulary=columna_de` ya que sin ellos, `TfidfVectorizer` aplicaría su propio `token_pattern`, que lo que hace es descartar tokens de un solo caracter como i o a, tambien separando los apostrofes, construyendo su propio vocabulario haciendo que las matrices dejen de ser comparables. Con el analizador que devuelve los tokens tal cual y el vocabulario fijo, el vectorizador ve los mismos tokens y las mismas columnas. También la aritmética de doble precisión tiene un error relativo de aproximadamente $10^{-16}$ y los pesos llegan a aproximadamente $10^{4}$, así que el ruido de redondeo queda por debajo de $10^{-11}$. En cambio, un error de fórmula sí se notaría, cómo por ejemplo, `smooth_idf=True` cambiaría el IDF de *alice* (DF = 3) de 2.79 a 2.56. Por eso $10^{-9}$ separa redondeo del error real.

Así se tiene entonces que la diferencia máxima es exactamente 0.0, porque ambas rutas multiplican el mismo conteo por el mismo IDF.

### 5. Representar consultas y calcular similitud coseno

Primero se definen cuatro consultas y se describe que busca una persona con cada una. Se cuentan las apariciones de cada palabra de la consulta y se las pondera con el IDF calculado a partir de los libros. Las palabras que no pertenecen al vocabulario se omiten porque la consulta no se incorpora al corpus


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

consultas = [
    {'consulta': 'whale ship sea',
     'necesidad': 'Encontrar relatos sobre ballenas y navegacion maritima.'},
    {'consulta': 'alice rabbit wonderland',
     'necesidad': 'Encontrar la historia de Alicia y el conejo en Wonderland.'},
    {'consulta': 'emma marriage love',
     'necesidad': 'Encontrar una novela sobre Emma y sus relaciones amorosas.'},
    {'consulta': 'king god israel',
     'necesidad': 'Encontrar pasajes religiosos sobre reyes e Israel.'},
]

def vector_consulta_manual(tokens):
    valores = np.zeros(len(terminos))
    for token in tokens:
        if token in columna_de:
            columna = columna_de[token]
            valores[columna] += idf_por_termino[token]
    return csr_matrix(valores.reshape(1, -1))

Con la similitud coseno se divide el producto punto entre el vector de un libro y el de la consulta por el producto de sus normas. Un valor mayor indica que sus términos ponderados apuntan en una dirección más parecida, donde si alguna norma vale cero, se asigna similitud cero. Para cada consulta, se compara el vector y los cosenos calculados manualmente contra los de scikit-learn y se muestra las diferencias máximas.


In [6]:
def cosenos_manuales(matriz, consulta):
    productos = (matriz @ consulta.T).toarray().ravel()
    cuadrados_docs = matriz.multiply(matriz).sum(axis=1)
    normas_docs = np.sqrt(np.asarray(cuadrados_docs).ravel())
    norma_consulta = float(np.sqrt(consulta.multiply(consulta).sum()))

    scores = np.zeros(matriz.shape[0])
    for i in range(matriz.shape[0]):
        denominador = normas_docs[i] * norma_consulta
        if denominador > 0:
            scores[i] = productos[i] / denominador
    return scores

resultados_consultas = []
for caso in consultas:
    tokens_q = normalizar_tokenizar(caso['consulta'])
    q_manual = vector_consulta_manual(tokens_q)
    q_sklearn = vectorizador.transform([tokens_q]) 
    diferencia_q = float(abs(q_manual - q_sklearn).max())

    scores_manual = cosenos_manuales(matriz_manual, q_manual)
    scores_sklearn = cosine_similarity(q_sklearn, matriz_sklearn).ravel()
    diferencia_coseno = float(np.max(np.abs(scores_manual - scores_sklearn)))
    if diferencia_q > tolerancia:
        raise ValueError('No coincide la consulta')
    if diferencia_coseno > tolerancia:
        raise ValueError('No coincide el coseno')

    resultados_consultas.append({
        'consulta': caso['consulta'], 'necesidad': caso['necesidad'],
        'tokens': tokens_q, 'q_manual': q_manual, 'scores': scores_manual,
    })
    print(caso['consulta'], '->', caso['necesidad'])
    print('  Diferencia del vector:', diferencia_q,
          '| diferencia maxima de coseno:', diferencia_coseno)

whale ship sea -> Encontrar relatos sobre ballenas y navegacion maritima.
  Diferencia del vector: 0.0 | diferencia maxima de coseno: 2.3869795029440866e-15
alice rabbit wonderland -> Encontrar la historia de Alicia y el conejo en Wonderland.
  Diferencia del vector: 0.0 | diferencia maxima de coseno: 3.885780586188048e-16
emma marriage love -> Encontrar una novela sobre Emma y sus relaciones amorosas.
  Diferencia del vector: 0.0 | diferencia maxima de coseno: 1.7486012637846216e-15
king god israel -> Encontrar pasajes religiosos sobre reyes e Israel.
  Diferencia del vector: 0.0 | diferencia maxima de coseno: 1.1379786002407855e-15


Cada consulta responde a una necesidad distinta y mezcla términos poco frecuentes con términos frecuentes, para ver el efecto del IDF dentro de un mismo vector. Se omiten las palabras fuera del vocabulario porque no tienen columna, haciendo que su aporte al producto punto sería 0. Solo cambiarían $\|q\|$, que es la misma constante para todos los libros de una consulta y por tanto no altera el orden de este.

Se tiene que la diferencia máxima entre el coseno calculado manualmente y el de scikit-learn es 2.4 × $10^{-15}$, que está muy por debajo de la tolerancia de 10⁻⁹. La única diferencia es el redondeo por distinto orden de las sumas y no por la fórmula.

Luego, se toma la primera consulta y el documento con la mayor similitud. También se muestra los términos que comparten, el peso TF-IDF de cada lado y cada producto parcial. Finalmente se calcula ambas normas y la similitud, para que se vea de dónde sale la puntuación en un ejemplo real.


In [7]:
caso = resultados_consultas[0]
indice_doc = int(np.argmax(caso['scores']))
doc_id = identificadores[indice_doc]
q = caso['q_manual']

filas_productos = []
for termino in caso['tokens']:
    if termino in tf_por_documento[doc_id]:
        columna = columna_de[termino]
        peso_q = float(q[0, columna])
        peso_d = float(matriz_manual[indice_doc, columna])
        filas_productos.append({
            'Termino compartido': termino, 'Peso de consulta': peso_q,
            'Peso de documento': peso_d, 'Producto parcial': peso_q * peso_d,
        })

print('Desglose:', caso['consulta'], 'con', doc_id, '(' + titulo_de(doc_id) + ')')
display(pd.DataFrame(filas_productos).round(6))
producto_punto = 0.0
for fila in filas_productos:
    producto_punto += fila['Producto parcial']

norma_q = float(np.sqrt(q.multiply(q).sum()))
fila_doc = matriz_manual.getrow(indice_doc)
norma_d = float(np.sqrt(fila_doc.multiply(fila_doc).sum()))
similitud = producto_punto / (norma_q * norma_d)
print('Producto punto:', round(producto_punto, 8))
print('Normas: consulta =', round(norma_q, 8), ', documento =', round(norma_d, 8))
print('Similitud coseno:', round(similitud, 8))

Desglose: whale ship sea con melville-moby_dick.txt (Moby Dick)


,Termino compartido,Peso de consulta,Peso de documento,Producto parcial
0,whale,2.098612,2310.572130,4848.995066
1,ship,1.325422,580.535011,769.454108
2,sea,1.117783,507.473498,567.245267


Producto punto: 6185.69444115
Normas: consulta = 2.72219709 , documento = 21338.77412695
Similitud coseno: 0.1064877


In [8]:
doc_ejemplo = identificadores[indice_doc]
fila_doc = matriz_manual.getrow(indice_doc)
cuadrados = {}
for columna, peso in zip(fila_doc.indices, fila_doc.data):
    cuadrados[terminos[columna]] = float(peso) * float(peso)
total = sum(cuadrados.values())

print('Libro:', doc_ejemplo, '| norma =', round(total ** 0.5, 2))
mayores = sorted(cuadrados.items(), key=lambda par: -par[1])[:6]
for termino, valor in mayores:
    print(' ', termino, '| peso =', round(valor ** 0.5, 1),
          '| % de la norma al cuadrado =', round(100 * valor / total, 1))

suma_consulta = 0.0
for termino in set(caso['tokens']):
    if termino in cuadrados:
        suma_consulta += cuadrados[termino]
print('Terminos de la consulta (whale, ship, sea): %.1f %% de la norma' %
      (100 * (suma_consulta ** 0.5) / (total ** 0.5)))

Libro: melville-moby_dick.txt | norma = 21338.77
  the | peso = 14431.0 | % de la norma al cuadrado = 45.7
  of | peso = 6609.0 | % de la norma al cuadrado = 9.6
  and | peso = 6430.0 | % de la norma al cuadrado = 9.1
  a | peso = 4728.0 | % de la norma al cuadrado = 4.9
  to | peso = 4625.0 | % de la norma al cuadrado = 4.7
  in | peso = 4172.0 | % de la norma al cuadrado = 3.8
Terminos de la consulta (whale, ship, sea): 11.4 % de la norma


El producto punto es 6,185.69 y se reparte así:
- *whale* 4,849.00 (78.4 %) 
- *ship* 769.45 (12.4 %)
- *sea* 567.25 (9.2 %) 

El producto parcial es "peso en la consulta" × "peso en el libro" = IDF × (TF × IDF) = TF × IDF², por lo que la rareza cuenta dos veces (una por el lado de la consulta y otra por el del libro). Por eso el término más raro, *whale*, con IDF 2.10 y TF 1,101, aporta casi cuatro quintos del total como se aprecia.

### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

Se ordena cada consulta según su similitud coseno y se presentan los cinco libros con mejor puntaje. La función de fragmentos busca una palabra de la consulta en el texto original para mostrar algo de contexto legible, mientras que el orden de entrada resuelve los empates. Al final se identifica qué términos comparten el primer resultado con la consulta


In [9]:
def fragmento_original(doc_id, tokens_q, ancho=190):
    texto = textos_originales[doc_id]
    inicio = 0
    for termino in tokens_q:
        patron = r'\b' + re.escape(termino) + r'\b'
        coincidencia = re.search(patron, texto, flags=re.IGNORECASE)
        if coincidencia is not None:
            inicio = max(0, coincidencia.start() - 60)
            break

    fragmento = ' '.join(texto[inicio:inicio + ancho].split())
    if inicio > 0:
        fragmento = '...' + fragmento
    return fragmento + '...'

for caso in resultados_consultas:
    ranking = np.argsort(-caso['scores'], kind='stable')[:5]
    filas_ranking = []
    for posicion, indice in enumerate(ranking, start=1):
        doc_id = identificadores[indice]
        filas_ranking.append({
            'Posicion': posicion,
            'Identificador': doc_id,
            'Titulo': titulo_de(doc_id),
            'Puntuacion': float(caso['scores'][indice]),
            'Fragmento': fragmento_original(doc_id, caso['tokens']),
        })

    print('\nConsulta:', caso['consulta'], '-', caso['necesidad'])
    display(pd.DataFrame(filas_ranking).round({'Puntuacion': 6}))
    primero = identificadores[ranking[0]]
    compartidos = set(caso['tokens']).intersection(tf_por_documento[primero])
    print('Interpretacion:', titulo_de(primero), 'encabeza el ranking y comparte',
          sorted(compartidos), 'con la consulta.')


Consulta: whale ship sea - Encontrar relatos sobre ballenas y navegacion maritima.


,Posicion,Identificador,Titulo,Puntuacion,Fragmento
0,1,melville-moby_dick.txt,Moby Dick,0.106488,"...in hand to school others, and to teach them..."
1,2,whitman-leaves.txt,Leaves of Grass,0.010366,"...eaves down with a strong arm, The mate stan..."
2,3,bryant-stories.txt,Stories to Tell to Children,0.007415,...icked him out again. HOW BROTHER RABBIT FOO...
3,4,chesterton-ball.txt,The Ball and The Cross,0.006365,"...was paid L12,000 a year for pretending to b..."
4,5,chesterton-brown.txt,The Wisdom of Father Brown,0.005395,...o alter the equilibrium; the whole coach he...


Interpretacion: Moby Dick encabeza el ranking y comparte ['sea', 'ship', 'whale'] con la consulta.

Consulta: alice rabbit wonderland - Encontrar la historia de Alicia y el conejo en Wonderland.


,Posicion,Identificador,Titulo,Puntuacion,Fragmento
0,1,carroll-alice.txt,Alice's Adventures in Wonderland,0.234619,[Alice's Adventures in Wonderland by Lewis Car...
1,2,burgess-busterbrown.txt,The Adventures of Buster Bear,0.005628,...He felt that he had the greatest news to te...
2,3,bryant-stories.txt,Stories to Tell to Children,0.001801,...unless somebody picked him out again. HOW B...
3,4,chesterton-thursday.txt,The Man Who Was Thursday,0.000774,"...th the moon; but in each case it was, someh..."
4,5,chesterton-ball.txt,The Ball and The Cross,0.000439,"...e sea! Shoo! Shoo! Shoo!"" The little man pl..."


Interpretacion: Alice's Adventures in Wonderland encabeza el ranking y comparte ['alice', 'rabbit', 'wonderland'] con la consulta.

Consulta: emma marriage love - Encontrar una novela sobre Emma y sus relaciones amorosas.


,Posicion,Identificador,Titulo,Puntuacion,Fragmento
0,1,austen-emma.txt,Emma,0.147954,[Emma by Jane Austen 1816] VOLUME I CHAPTER I ...
1,2,blake-poems.txt,Poems,0.016165,"...the new-born infant's tear, And blights wit..."
2,3,whitman-leaves.txt,Leaves of Grass,0.007748,...been with us that day round the chowder-ket...
3,4,milton-paradise.txt,Paradise Lost,0.005799,"...irit, that deigned To travel with Tobias, a..."
4,5,austen-sense.txt,Sense and Sensibility,0.004636,...the children added a relish to his existenc...


Interpretacion: Emma encabeza el ranking y comparte ['emma', 'love', 'marriage'] con la consulta.

Consulta: king god israel - Encontrar pasajes religiosos sobre reyes e Israel.


,Posicion,Identificador,Titulo,Puntuacion,Fragmento
0,1,bible-kjv.txt,The King James Bible,0.074608,[The King James Bible] The Old Testament of th...
1,2,shakespeare-hamlet.txt,The Tragedie of Hamlet,0.036230,...answer me: Stand & vnfold your selfe Bar. L...
2,3,milton-paradise.txt,Paradise Lost,0.020837,"...in dreadful deeds Fearless, endangered Heav..."
3,4,shakespeare-macbeth.txt,The Tragedie of Macbeth,0.020061,...lthie ayre. Exeunt. Scena Secunda. Alarum w...
4,5,bryant-stories.txt,Stories to Tell to Children,0.018105,...in the jungle there was a great big Lion; a...


Interpretacion: The King James Bible encabeza el ranking y comparte ['god', 'israel', 'king'] con la consulta.


In [10]:
for caso in resultados_consultas:
    ranking = np.argsort(-caso['scores'], kind='stable')[:5]
    terminos_q = list(dict.fromkeys(caso['tokens']))
    filas_tf = []
    for posicion, indice in enumerate(ranking, start=1):
        doc_id = identificadores[indice]
        fila = {'Posicion': posicion, 'Identificador': doc_id, 'Longitud': longitudes[doc_id]}
        for termino in terminos_q:
            fila['TF ' + termino] = tf_por_documento[doc_id][termino]
        fila['Puntuacion'] = round(float(caso['scores'][indice]), 6)
        filas_tf.append(fila)
    print('\nConsulta:', caso['consulta'])
    display(pd.DataFrame(filas_tf))


Consulta: whale ship sea


,Posicion,Identificador,Longitud,TF whale,TF ship,TF sea,Puntuacion
0,1,melville-moby_dick.txt,216201,1101,438,454,0.106488
1,2,whitman-leaves.txt,124335,5,75,190,0.010366
2,3,bryant-stories.txt,46321,16,5,20,0.007415
3,4,chesterton-ball.txt,82245,2,38,57,0.006365
4,5,chesterton-brown.txt,72391,0,17,62,0.005395



Consulta: alice rabbit wonderland


,Posicion,Identificador,Longitud,TF alice,TF rabbit,TF wonderland,Puntuacion
0,1,carroll-alice.txt,26688,386,47,3,0.234619
1,2,burgess-busterbrown.txt,15969,0,11,0,0.005628
2,3,bryant-stories.txt,46321,0,10,0,0.001801
3,4,chesterton-thursday.txt,58300,1,1,1,0.000774
4,5,chesterton-ball.txt,82245,0,4,0,0.000439



Consulta: emma marriage love


,Posicion,Identificador,Longitud,TF emma,TF marriage,TF love,Puntuacion
0,1,austen-emma.txt,160999,786,35,117,0.147954
1,2,blake-poems.txt,6839,0,1,29,0.016165
2,3,whitman-leaves.txt,124335,0,8,272,0.007748
3,4,milton-paradise.txt,80089,0,3,99,0.005799
4,5,austen-sense.txt,119965,0,44,77,0.004636



Consulta: king god israel


,Posicion,Identificador,Longitud,TF king,TF god,TF israel,Puntuacion
0,1,bible-kjv.txt,852256,2259,4446,2565,0.074608
1,2,shakespeare-hamlet.txt,29698,171,26,1,0.036230
2,3,milton-paradise.txt,80089,48,245,4,0.020837
3,4,shakespeare-macbeth.txt,17833,54,14,0,0.020061
4,5,bryant-stories.txt,46321,125,24,11,0.018105


En las cuatro consultas gana el libro esperado (*Moby Dick*, *Alice*, *Emma*, la Biblia) y la interpretación impresa muestra que contiene todos los términos de la consulta. El ganador tiene TF muy alto en los términos raros como *whale* 1,101 o *alice* 386. 

Se tiene también que los puestos 2 a 5 son coincidencias débiles, eso ya que esos libros mencionan las palabras raras muy pocas veces, cómo por ejemplo *whale* de 0 a 16 veces frente a 1,101 en *Moby Dick*.

De forma similar, *Hamlet* queda segundo en `king god israel` porque tiene *king* 171 veces y *god* 26 veces, contrastando con *israel* que tiene solo una aparición. Coincide en palabras y no en el tema, por eso su puesto, ya que el método compara palabras y no el significado.